# Лекция 03. Алгоритмы и сложность

Учимся сравнивать решения до запуска программы и разбираем два базовых способа поиска.

## Цели

После лекции вы сможете:

- определить размер входа и интересующий ресурс;
- различать время работы и дополнительную память;
- читать оценки `O`, `Ω` и `Θ`;
- различать лучший и худший случаи;
- реализовать линейный и бинарный поиск;
- объяснить предусловие бинарного поиска;
- оценить сложность поиска;
- корректно провести небольшой вычислительный эксперимент.

## Перед началом

Понадобятся списки, индексы, циклы, условия и функции из первых двух занятий. В примерах считаем, что доступ к элементу списка по индексу и сравнение двух обычных чисел занимают постоянное время.

Анализ сложности не заменяет измерения, а измерения не заменяют анализ: сегодня научимся использовать оба инструмента вместе.

## Зачем оценивать алгоритм

Два решения могут возвращать одинаковый ответ, но по-разному вести себя при росте данных. На списке из десяти элементов разница незаметна; на миллионах элементов один алгоритм может закончить работу, а другой — нет.

Нас интересует не точное число секунд на конкретном ноутбуке, а **как растут затраты вместе с размером входа**. Это позволяет сравнивать идеи независимо от процессора, языка и фоновой нагрузки.

## Размер входа и базовая операция

Сначала выбирают параметр размера. Для поиска в списке это обычно `N = len(values)`. Затем определяют операцию, число выполнений которой отражает основную работу: например, сравнение элемента с целью.

В задачах с несколькими входами параметров может быть несколько: `N` строк и `M` столбцов, `V` вершин и `E` рёбер. Нельзя автоматически называть всё `N`, не объяснив его смысл.

In [ ]:
def count_linear_search_comparisons(
    values: list[int], target: int
) -> int:
    comparisons = 0
    for value in values:
        comparisons += 1
        if value == target:
            return comparisons
    return comparisons


values = [4, 8, 15, 16, 23, 42]
print(count_linear_search_comparisons(values, 4))
print(count_linear_search_comparisons(values, 42))
print(count_linear_search_comparisons(values, 100))

## Время и память

Обычно оценивают два ресурса:

- **время** — число существенных операций;
- **дополнительную память** — объём памяти сверх входных данных и результата.

Один алгоритм может экономить время ценой памяти. Например, можно один раз создать отсортированную копию данных, а затем выполнять много бинарных поисков. Копия требует `Θ(N)` дополнительной памяти, а подготовка тоже занимает время. Для единственного запроса она может не окупиться, а для тысяч запросов — окупиться.

In [ ]:
values = [40, 10, 30, 20]
ordered = sorted(values)  # Создан новый список.

print(values)
print(ordered)
print(ordered is values)

## Асимптотическая нотация

Пусть `T(N)` — число операций на входе размера `N`.

- `T(N) = O(f(N))` задаёт асимптотическую **верхнюю границу**: начиная с некоторого размера рост не быстрее `f(N)` с точностью до постоянного множителя.
- `T(N) = Ω(f(N))` задаёт асимптотическую **нижнюю границу**.
- `T(N) = Θ(f(N))` задаёт **точный порядок роста**: одновременно верны верхняя и нижняя границы одного порядка.

В прикладных обсуждениях словом «сложность» часто называют наиболее полезную тесную оценку `Θ`, хотя записывают её как `O`. В строгом рассуждении эти обозначения различаются.

## Типичные порядки роста

| Порядок | Название | Пример | При увеличении `N` вдвое |
| --- | --- | --- | --- |
| `Θ(1)` | постоянный | доступ `values[index]` | почти без изменения |
| `Θ(log N)` | логарифмический | бинарный поиск | примерно +1 шаг |
| `Θ(N)` | линейный | полный проход | примерно ×2 |
| `Θ(N log N)` | линейно-логарифмический | эффективная сортировка | немного больше ×2 |
| `Θ(N²)` | квадратичный | все пары элементов | примерно ×4 |
| `Θ(2ᴺ)` | экспоненциальный | полный перебор подмножеств | возведение времени в квадрат |

Основание логарифма в асимптотике обычно не указывают: логарифмы с разными постоянными основаниями отличаются постоянным множителем.

In [ ]:
from math import log2

for size in [8, 16, 32, 64]:
    print(
        f"N={size:>2}",
        f"log2(N)={log2(size):>3.0f}",
        f"N²={size ** 2:>4}",
    )

## Константы и младшие слагаемые

При асимптотическом анализе сохраняют доминирующий порядок роста:

- `3N + 20 = Θ(N)`;
- `N² + 100N = Θ(N²)`;
- `log₂N + 7 = Θ(log N)`.

Это не означает, что константы не важны на практике. Алгоритм с `100N` операций может проиграть алгоритму с `N²` на небольших входах. Асимптотика отвечает на вопрос о поведении при росте `N`, а не объявляет победителя для любого размера.

## Лучший, худший и средний случаи

Для одного размера входа время может зависеть от расположения данных. У линейного поиска:

- лучший случай — цель стоит первой: `Θ(1)`;
- худший случай — цель последняя или отсутствует: `Θ(N)`;
- средний случай требует модели вероятностей: какие входы и цели считаются вероятными.

Если модель не задана, безопаснее явно анализировать худший случай. Фраза «в среднем быстро» без описания распределения входов ничего не доказывает.

## Линейный поиск

Линейный поиск просматривает элементы слева направо и останавливается при первом совпадении. Он не требует сортировки и работает с любым перебираемым набором данных.

После проверки первых `k` элементов мы знаем: среди них цели нет. Это простое утверждение является **инвариантом** цикла и помогает обосновать корректность.

In [ ]:
def linear_search(values: list[int], target: int) -> int:
    for index, value in enumerate(values):
        if value == target:
            return index
    return -1


print(linear_search([7, 2, 7, 9], 7))
print(linear_search([7, 2, 7, 9], 5))

## Сложность линейного поиска

В лучшем случае выполняется одно сравнение: `Θ(1)`. В худшем — `N` сравнений: `Θ(N)`. Дополнительная память равна `Θ(1)`: функция хранит только текущий индекс и элемент.

Ранний `return` улучшает отдельные запуски, но не меняет оценку худшего случая. Наличие короткого пути не позволяет назвать весь алгоритм константным.

## Бинарный поиск: цена ускорения

Если список **отсортирован**, одно сравнение позволяет отбросить примерно половину оставшихся элементов. Если `values[middle] < target`, слева от `middle` цели быть не может; иначе можно отбросить правую часть.

Упорядоченность — предусловие алгоритма, а не небольшая оптимизация. На неотсортированном списке бинарный поиск может вернуть правдоподобный, но неверный результат.

## Полуинтервал поиска `[left, right)`

Будем хранить ещё не исключённые позиции в полуинтервале `[left, right)`: левая граница включена, правая не включена. В начале `left = 0`, `right = len(values)`. Пока `left < right`, интервал непуст.

Средний индекс `middle = (left + right) // 2`. На каждом шаге одна из границ сдвигается так, чтобы `middle` больше не оставался в следующем интервале. Это гарантирует завершение.

In [ ]:
def binary_search(values: list[int], target: int) -> int:
    left = 0
    right = len(values)

    while left < right:
        middle = (left + right) // 2
        if values[middle] == target:
            return middle
        if values[middle] < target:
            left = middle + 1
        else:
            right = middle

    return -1


print(binary_search([2, 5, 8, 12, 16, 23, 38], 16))
print(binary_search([2, 5, 8, 12, 16, 23, 38], 10))

## Трассировка бинарного поиска

Для цели `16` в списке `[2, 5, 8, 12, 16, 23, 38]` интервалы меняются так:

| `left` | `right` | `middle` | значение | действие |
| ---: | ---: | ---: | ---: | --- |
| 0 | 7 | 3 | 12 | отбросить позиции `0..3` |
| 4 | 7 | 5 | 23 | отбросить позиции `5..6` |
| 4 | 5 | 4 | 16 | цель найдена |

Таблица границ полезнее угадывания кода: она быстро обнаруживает зацикливание и ошибки на единицу.

## Повторяющиеся значения

Простой бинарный поиск возвращает **какое-нибудь** совпадение. Если нужен первый индекс, при равенстве нельзя сразу завершаться: найденная позиция становится кандидатом, а поиск продолжается слева.

Более общий подход — искать первую позицию, на которой значение не меньше цели. Такая граница называется `lower bound`. После поиска остаётся проверить, действительно ли на найденной позиции стоит цель.

In [ ]:
def lower_bound(values: list[int], target: int) -> int:
    left = 0
    right = len(values)

    while left < right:
        middle = (left + right) // 2
        if values[middle] < target:
            left = middle + 1
        else:
            right = middle

    return left


values = [1, 2, 2, 2, 5]
for target in [0, 2, 3, 6]:
    print(target, lower_bound(values, target))

## Почему получается `Θ(log N)`

После `k` шагов от `N` кандидатов остаётся примерно `N / 2ᵏ`. Поиск заканчивается, когда остаётся не больше одного кандидата:

`N / 2ᵏ <= 1`, следовательно, `k >= log₂N`.

Поэтому бинарный поиск выполняет `Θ(log N)` сравнений в худшем случае и использует `Θ(1)` дополнительной памяти в итеративной реализации. Для миллиарда элементов достаточно примерно 30 делений диапазона пополам.

In [ ]:
size = 1
for steps in range(0, 31, 5):
    if steps > 0:
        size = 2 ** steps
    print(f"{steps:>2} шагов -> до {size:,} элементов")

## Как выбрать поиск

| Ситуация | Подход | Стоимость запроса |
| --- | --- | --- |
| данные не отсортированы, один запрос | линейный поиск | `Θ(N)` |
| данные уже отсортированы | бинарный поиск | `Θ(log N)` |
| данные не отсортированы, нужны исходные индексы | линейный поиск | `Θ(N)` |
| много запросов к неизменным данным | один раз подготовить отсортированную копию | запрос `Θ(log N)` после подготовки |
| много запросов диапазона | сортировка и бинарные границы | запрос `Θ(log N)` после подготовки |

Бинарный поиск не делает сортировку бесплатной. Если ради одного запроса сначала сортировать данные, общая стоимость останется `Θ(N log N)` и может измениться порядок элементов. Алгоритмы сортировки подробно разберём на занятии 6.

## Стандартная библиотека `bisect`

В рабочем коде границы отсортированного списка обычно ищут модулем `bisect`. `bisect_left(values, target)` возвращает первую позицию, куда можно вставить цель, сохранив порядок; `bisect_right` — позицию после всех равных элементов.

Собственную реализацию пишем, чтобы понять инвариант и границы. После этого стандартная функция уменьшает риск ошибки и яснее выражает намерение.

In [ ]:
from bisect import bisect_left, bisect_right

values = [1, 2, 2, 2, 5]
left = bisect_left(values, 2)
right = bisect_right(values, 2)
print(left, right, right - left)

## Измерение времени

Для небольшого эксперимента используют `time.perf_counter()`, повторяют операцию много раз и сравнивают входы разных размеров. Важно:

- измерять одну и ту же задачу;
- отделять подготовку данных от запроса или явно включать её в обе стратегии;
- повторять измерения, потому что система шумит;
- не делать общий вывод по одному маленькому входу;
- сначала проверить корректность сравниваемых функций.

Время в секундах зависит от машины, а характер роста должен быть устойчивее.

In [ ]:
from bisect import bisect_left
from time import perf_counter

values = list(range(100_000))
targets = [-1] * 100

started = perf_counter()
for target in targets:
    target in values
linear_seconds = perf_counter() - started

started = perf_counter()
for target in targets:
    position = bisect_left(values, target)
    position < len(values) and values[position] == target
binary_seconds = perf_counter() - started

print(f"linear: {linear_seconds:.6f} s")
print(f"binary: {binary_seconds:.6f} s")

## Скрытая стоимость памяти

Оценка должна учитывать временные объекты. Выражение `values[1:]` создаёт новый список длины `N - 1`, поэтому цикл `for value in values[1:]:` использует `Θ(N)` дополнительной памяти. Обход по индексам или прямой обход исходного списка копии не создаёт.

Аналогично `sorted(values)` создаёт новый список. Иногда это правильная цена подготовки данных; важно назвать её явно.

In [ ]:
values = [10, 20, 30, 40]
tail = values[1:]
tail[0] = 999
print(values)
print(tail)

## Неожиданно, но по правилам

Перед запуском предскажите результат.

1. Бинарный поиск не проверяет предусловие сортировки. На неупорядоченном списке он не обязан сообщать об ошибке и может просто не найти существующее значение.
2. Один цикл не гарантирует линейное время. Если на каждой итерации создавать срез оставшейся части, суммарно копируется `N + (N - 1) + ... + 1 = Θ(N²)` элементов.
3. Сортировка копии разрешает бинарный поиск, но найденный индекс относится к отсортированной копии, а не к исходному списку.

In [ ]:
unsorted_values = [1, 4, 2, 5, 7]
print(binary_search(unsorted_values, 4))  # 4 есть, но поиск вернул -1.

def count_copied_items(size: int) -> int:
    values = list(range(size))
    copied_items = 0
    for index in range(size):
        tail = values[index:]
        copied_items += len(tail)
    return copied_items

print(count_copied_items(5), count_copied_items(10))

target = 4
ordered = sorted(unsorted_values)
position = binary_search(ordered, target)
print(ordered, position, unsorted_values.index(target))

## Самопроверка

1. Что означает `N` в анализе поиска по списку?
2. Чем время работы отличается от дополнительной памяти?
3. Чем `O`, `Ω` и `Θ` отличаются друг от друга?
4. Почему `3N + 20` имеет порядок `Θ(N)`?
5. Каковы лучший и худший случаи линейного поиска?
6. Какое предусловие требуется бинарному поиску?
7. Почему полуинтервал `[left, right)` удобен для границ?
8. Что должен делать бинарный поиск при равенстве, если нужен первый индекс?
9. Когда подготовка отсортированной копии может окупиться?
10. Почему один замер времени не доказывает асимптотическую оценку?
11. Как срез может изменить оценку времени и дополнительной памяти?

## Итоги

- Сложность описывает рост затрат вместе с размером входа.
- Время и дополнительную память оценивают отдельно.
- `O`, `Ω` и `Θ` задают верхнюю, нижнюю и тесную асимптотические границы.
- Лучший и худший случаи могут иметь разные порядки роста.
- Линейный поиск не требует порядка и работает за `Θ(N)` в худшем случае.
- Бинарный поиск требует отсортированных данных и работает за `Θ(log N)`.
- Подготовка индекса или сортировка имеют собственную стоимость.
- Эксперимент дополняет анализ, если корректность и условия измерения контролируются.

На [семинаре](seminar.ipynb) реализуем поиски, проверим границы и сравним рост времени экспериментально.